# Titanic Baseline

This notebook builds a small classification baseline for the Titanic dataset. The goal is to practice the classic ML workflow, not to maximize the Kaggle score on the first try.

## 1. Import Libraries

We use pandas for data handling, scikit-learn for preprocessing and modeling, and matplotlib for a simple evaluation plot.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## 2. Load the Data

Place the Titanic training file at `01_titanic/data/train.csv`. The path cell below works whether the notebook runs from this folder or from the repository root.

In [ ]:
DATA_PATH = Path("data/train.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("01_titanic/data/train.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError("Place train.csv in 01_titanic/data/ before running this notebook.")

titanic = pd.read_csv(DATA_PATH)
titanic.head()

## 3. Choose a Small Feature Set

Start with a few readable columns. Later, you can add more features and compare the result.

In [ ]:
target_column = "Survived"
numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Sex", "Embarked"]
feature_columns = numeric_features + categorical_features

X = titanic[feature_columns]
y = titanic[target_column]

X.head()

## 4. Split into Train and Test Sets

The model learns from the training rows. The test rows are held back for evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

len(X_train), len(X_test)

## 5. Build the Preprocessing and Model Pipeline

Numeric columns get missing values filled with the median. Categorical columns get missing values filled with the most common value, then one-hot encoded.

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000)),
    ]
)

## 6. Train and Evaluate

Accuracy is a useful first metric here, but also read precision and recall so you understand which class is harder for the model.

In [ ]:
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, predictions):.3f}")
print(classification_report(y_test, predictions))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, predictions)
plt.title("Titanic Baseline Confusion Matrix")
plt.tight_layout()
plt.show()

## Next Experiments

- Add `Cabin` carefully by extracting whether it is missing.
- Try `RandomForestClassifier` and compare metrics.
- Use cross-validation after the simple train/test split feels clear.